# Walmart Pipeline — Verification Notebook

One-stop self-check for the pipeline's output. Run after a fresh DAG run or a
`--full-refresh` to confirm all layers are consistent.

| Layer | Expected |
|---|---|
| Bronze | orders=10,000 · customers=2,000 · products=500 · order_items=30,021 · stores=25 · employees=250 |
| Silver_t | same as Bronze (deduplicated) |
| Silver_b (OBT) | 30,021 |
| Gold | dim_customers=2,000 · dim_products=500 · dim_stores=25 · dim_employees=250 · fact_order_items=30,021 · fact_orders=10,000 |
| Snapshots | same as Gold dims (first run) |

In [0]:
%sql
-- Bronze row counts
select 'orders' as table_name, count(*) as row_count from walmart.bronze.orders
union all
select 'customers', count(*) from walmart.bronze.customers
union all
select 'products', count(*) from walmart.bronze.products
union all
select 'order_items', count(*) from walmart.bronze.order_items
union all
select 'stores', count(*) from walmart.bronze.stores
union all
select 'employees', count(*) from walmart.bronze.employees
order by table_name;

table_name,row_count
customers,2000
employees,250
order_items,30021
orders,10000
products,500
stores,25


In [0]:
%sql
-- Silver layer: 6 _t tables + OBT
select 'customers_t'    as table_name, count(*) as row_count from walmart.silver_t.customers_t
union all
select 'employees_t',    count(*) from walmart.silver_t.employees_t
union all
select 'order_items_t',  count(*) from walmart.silver_t.order_items_t
union all
select 'orders_t',       count(*) from walmart.silver_t.orders_t
union all
select 'products_t',     count(*) from walmart.silver_t.products_t
union all
select 'stores_t',       count(*) from walmart.silver_t.stores_t
union all
select 'obt_b',          count(*) from walmart.silver_b.obt_b
order by table_name;

table_name,row_count
customers_t,2000
employees_t,250
obt_b,30021
order_items_t,30021
orders_t,10000
products_t,500
stores_t,25


In [0]:
%sql
-- Gold row counts (2 facts + 4 dims)
select 'dim_customers'   as gold_table, count(*) as gold_rows from walmart.gold.dim_customers
union all
select 'dim_employees',   count(*) from walmart.gold.dim_employees
union all
select 'dim_products',    count(*) from walmart.gold.dim_products
union all
select 'dim_stores',      count(*) from walmart.gold.dim_stores
union all
select 'fact_order_items', count(*) from walmart.gold.fact_order_items
union all
select 'fact_orders',     count(*) from walmart.gold.fact_orders
order by gold_table;

gold_table,gold_rows
dim_customers,2000
dim_employees,250
dim_products,500
dim_stores,25
fact_order_items,30021
fact_orders,10000


In [0]:
%sql
-- Snapshot row counts (should equal Gold dim counts on first run)
select 'customers' as tbl, count(*) from walmart.snapshots.dim_customers_snapshot
union all
select 'employees', count(*) from walmart.snapshots.dim_employees_snapshot
union all
select 'products',  count(*) from walmart.snapshots.dim_products_snapshot
union all
select 'stores',    count(*) from walmart.snapshots.dim_stores_snapshot;



tbl,count(*)
customers,2000
employees,250
products,500
stores,25


In [0]:
%sql
-- Dimension uniqueness: no key should appear more than once (all rows should be 0)
select 'dim_customers' as table_name, count(*) as duplicate_keys
from (select customer_id from walmart.gold.dim_customers group by 1 having count(*) > 1)
union all
select 'dim_products', count(*)
from (select product_id from walmart.gold.dim_products group by 1 having count(*) > 1)
union all
select 'dim_stores', count(*)
from (select store_id from walmart.gold.dim_stores group by 1 having count(*) > 1)
union all
select 'dim_employees', count(*)
from (select employee_id from walmart.gold.dim_employees group by 1 having count(*) > 1)
order by table_name;

table_name,duplicate_keys
dim_customers,0
dim_employees,0
dim_products,0
dim_stores,0


In [0]:
%sql
-- fact_orders grain check: order-grain (1 row per order_id)
-- Expected: total rows == distinct order_ids == 10,000
select
    count(*) as total_rows,
    count(distinct order_id) as distinct_orders
from walmart.gold.fact_orders;

total_rows,distinct_orders
10000,10000


In [0]:
%sql
-- fact_order_items grain check: line-item grain (1 row per order_item_id)
-- Expected: total rows == distinct order_item_ids == 30,021
select
    count(*) as total_rows,
    count(distinct order_item_id) as distinct_order_items
from walmart.gold.fact_order_items;

total_rows,distinct_order_items
30021,30021


In [0]:
%sql
select count(distinct customer_id) as customers_t_total from walmart.silver_t.customers_t;

customers_t_total
2000


In [0]:
%sql
select count(distinct customer_id) as obt_b_total from walmart.silver_b.obt_b;
-- Difference should be 9 (the customers with zero orders)

obt_b_total
1991


In [0]:
%sql
--  Key finding: customers in customers_t but not in obt_b (no orders)
select c.customer_id, c.first_name, c.last_name, c.email
from walmart.silver_t.customers_t c
left join (select distinct customer_id from walmart.silver_b.obt_b) o
    on c.customer_id = o.customer_id
where o.customer_id is null
order by c.customer_id;

customer_id,first_name,last_name,email
20,Keith,Jenkins,johnsonmelissa@example.net
255,Joshua,Le,starkmolly@example.net
413,Karen,Robinson,bautistamatthew@example.org
604,Ryan,Spencer,katherine45@example.net
712,Robert,Thomas,fmartin@example.com
743,Benjamin,Robinson,dianekelly@example.com
1345,Tammy,Coleman,jamesstokes@example.net
1443,Leah,Roberson,lisasantos@example.com
1725,Cole,Norris,christophermcgee@example.net


In [0]:
%sql
-- Cast validation: numeric / date columns in Bronze can be safely cast
-- Expected: bad_rows == 0 for all
select 'orders.total_amount'     as col, count(*) as bad_rows
from walmart.bronze.orders
where cast(total_amount as decimal(18,2)) is null and total_amount is not null
union all
select 'orders.order_timestamp', count(*)
from walmart.bronze.orders
where cast(order_timestamp as timestamp) is null and order_timestamp is not null
union all
select 'products.price', count(*)
from walmart.bronze.products
where cast(price as decimal(18,2)) is null and price is not null
union all
select 'order_items.unit_price', count(*)
from walmart.bronze.order_items
where cast(unit_price as decimal(18,2)) is null and unit_price is not null
union all
select 'order_items.line_amount', count(*)
from walmart.bronze.order_items
where cast(line_amount as decimal(18,2)) is null and line_amount is not null;

col,bad_rows
orders.total_amount,0
orders.order_timestamp,0
products.price,0
order_items.unit_price,0
order_items.line_amount,0


## Expected results

- All row counts match the table at the top.
- Dimension uniqueness: all `duplicate_keys = 0`.
- `fact_orders` is order-grain; `fact_order_items` is line-item grain.
- `customers_t` has 2,000 distinct customers; `obt_b` has 1,991 (9 fewer).
- Cast validation: all `bad_rows = 0`.